![silver-to-gold.jpg](./silver-to-gold.jpg "silver-to-gold.jpg")

In [0]:
# --- 0. Configuración de Rutas de Checkpoint (En tu Volumen) ---
base_volume = "/Volumes/ecomotive/landing/ecomotive_vol"

# Rutas para checkpoints (Sistema de archivos)
chk_gold_drivers = f"{base_volume}/sys/checkpoints/gold_drivers"
chk_gold_sensors = f"{base_volume}/sys/checkpoints/gold_sensors"
chk_gold_agg     = f"{base_volume}/sys/checkpoints/gold_agg_kpi" # Preparamos ya el de la agregación

# Nombres de las tablas destino
table_dim_drivers = "ecomotive.gold.dim_drivers"
table_fact_sensors = "ecomotive.gold.fact_sensors"
table_gold_agg = "ecomotive.gold.daily_driver_kpis"

print("--- INICIANDO LIMPIEZA DE ENTORNO GOLD ---")

# 1. Borrar tablas si existen (Metadatos y Datos gestionados)
spark.sql(f"DROP TABLE IF EXISTS {table_dim_drivers}")
spark.sql(f"DROP TABLE IF EXISTS {table_fact_sensors}")
spark.sql(f"DROP TABLE IF EXISTS {table_gold_agg}")

# 2. Borrar Checkpoints físicos (Para reiniciar el stream desde cero)
# dbutils.fs.rm borra recursivamente el directorio
dbutils.fs.rm(chk_gold_drivers, True)
dbutils.fs.rm(chk_gold_sensors, True)
dbutils.fs.rm(chk_gold_agg, True)

print("--- LIMPIEZA COMPLETADA: LISTO PARA EJECUTAR ---")

--- INICIANDO LIMPIEZA DE ENTORNO GOLD ---
--- LIMPIEZA COMPLETADA: LISTO PARA EJECUTAR ---


In [0]:
from pyspark.sql.functions import col, expr

# Leemos la fuente Silver (Stream)
df_silver_drivers = spark.readStream.table("ecomotive.silver.silver_drivers")

df_dim_drivers = df_silver_drivers.select(
    col("driver_id"),
    col("driver_name"),
    col("license_ref"),
    expr("try_cast(hourly_wage as double) as hourly_wage"), 
    col("certifications"), 
    col("hire_date")
)

(df_dim_drivers.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_gold_drivers)
    .trigger(availableNow=True)
    .toTable(table_dim_drivers)
)

In [0]:
from pyspark.sql.functions import col, expr

# Leemos la fuente Silver (Stream)
df_silver_sensors = spark.readStream.table("ecomotive.silver.silver_sensors")

df_fact_sensors = df_silver_sensors.select(
    col("event_id"),
    col("sensor_id"),
    col("driver_id"),
    col("event_time"),
    col("rpm"),
    col("temperature"),
    expr("try_cast(battery_level as double) as battery_level"),
    col("engine_status"),
    col("latitude"),
    col("longitude")
)

(df_fact_sensors.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", chk_gold_sensors)
    .trigger(availableNow=True)
    .toTable(table_fact_sensors)
)

In [0]:
%sql
SELECT 'DRIVERS' as type, count(*) as count FROM ecomotive.silver.silver_drivers
UNION ALL
SELECT 'SENSORS' as type, count(*) as count FROM ecomotive.silver.silver_sensors;

type,count
DRIVERS,200
SENSORS,251


In [0]:
%sql
SELECT 'DIMENSIONS' as type, count(*) as count FROM ecomotive.gold.dim_drivers
UNION ALL
SELECT 'FACTS' as type, count(*) as count FROM ecomotive.gold.fact_sensors;

type,count
DIMENSIONS,200
FACTS,251


##### Estrategia de Agregación
Dado que el nombre de la tabla que definimos en el paso 0 es daily_driver_kpis, agruparemos los datos por Día y por Conductor.
- Aplicamos broadcats JOIN: dado que la tabla de DIMENSIONES es pequeña, frente a la de HECHOS (Grande)

![broadcast_explain.jpg](./broadcast_explain.jpg "broadcast_explain.jpg")




In [0]:
from pyspark.sql.functions import col, window, avg, max, count, to_date, broadcast

# 1. Lectura de las fuentes
# Fact (Stream): Queremos procesar los nuevos datos que llegan a Gold
df_fact_stream = spark.readStream.table("ecomotive.gold.fact_sensors")

# Dim (Static): Leemos la dimensión actual como tabla estática (Delta Table)
df_dim_static = spark.read.table("ecomotive.gold.dim_drivers")

# 2. Join (Stream-Static)
# Unimos los eventos del sensor con la información del conductor
df_joined = df_fact_stream.join(
    broadcast(df_dim_static), 
    on="driver_id",
    how="inner"
)

# 3. Agregación (Cálculo de KPIs)
# Agrupamos por ventana de 1 día y por nombre del conductor
df_kpis = df_joined \
    .groupBy(
        window(col("event_time"), "1 day").alias("time_window"),
        col("driver_name")
    ) \
    .agg(
        avg("rpm").alias("avg_rpm"),
        max("temperature").alias("max_temp"),
        avg("battery_level").alias("avg_battery"),
        count("event_id").alias("total_events")
    ) \
    .select(
        col("time_window.start").alias("date"), # Extraemos la fecha de la ventana
        col("driver_name"),
        col("avg_rpm"),
        col("max_temp"),
        col("avg_battery"),
        col("total_events")
    )

# 4. Escritura (Sink)
# Usamos 'complete' para actualizar los KPIs existentes con la nueva info
(df_kpis.writeStream
    .format("delta")
    .outputMode("complete") 
    .option("checkpointLocation", chk_gold_agg)
    .trigger(availableNow=True)
    .toTable(table_gold_agg)
)

In [0]:
%sql
SELECT * FROM ecomotive.gold.daily_driver_kpis
ORDER BY date DESC, total_events DESC;

date,driver_name,avg_rpm,max_temp,avg_battery,total_events
2025-10-01T00:00:00.000Z,Luis Garcia,1856.851851851852,96.0,53.989999999999995,27
2025-10-01T00:00:00.000Z,Juan Alvarez,1803.8823529411766,97.7,39.728235294117646,17
2025-10-01T00:00:00.000Z,Sofia Gomez,1894.875,96.7,56.203125,16
2025-10-01T00:00:00.000Z,Ana Hernandez,1993.3333333333333,97.0,57.172666666666665,15
2025-10-01T00:00:00.000Z,David Navarro,1977.2666666666667,97.4,41.118,15
2025-10-01T00:00:00.000Z,Roberto Smith,2227.3333333333335,97.5,44.76500000000001,12
2025-10-01T00:00:00.000Z,Ana Romero,1821.75,97.6,60.72416666666666,12
2025-10-01T00:00:00.000Z,Elena Romero,1892.4545454545455,96.0,52.81454545454546,11
2025-10-01T00:00:00.000Z,Laura Navarro,2239.2727272727275,97.1,42.004000000000005,11
2025-10-01T00:00:00.000Z,Ana Lopez,1815.8,97.9,61.821000000000005,10


In [0]:
%sql
SELECT SUM(total_events) AS total_eventos
 FROM ecomotive.gold.daily_driver_kpis

total_eventos
251
